# Shimi Poker — רינדור מחדש של סצינה 4 (החזיה) בקולאב
**הוראות:** Runtime → Change runtime type → T4 GPU → Save. אחר כך Runtime → Run all. הכל אוטומטי.
עוגן: שמלה+ידידה (e8383ce46). פרומפט נעול מהמאושר של סאן (חברה במקום אח, מסתיים בחזיה+תחתונים).

In [ ]:
# 1) בדיקת GPU + התקנות
import torch
assert torch.cuda.is_available(), 'אין GPU! Runtime → Change runtime type → T4 GPU'
print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
import subprocess
subprocess.run(['pip','install','-q','-U','git+https://github.com/huggingface/diffusers.git','transformers>=4.51','accelerate','ftfy','protobuf','imageio[ffmpeg]','safetensors','requests'], check=True)
print('התקנות הושלמו')

In [ ]:
# 2) טעינת Wan 2.2 TI2V-5B — מקודד טקסט bf16 על CPU (ללא NaN), טרנספורמר fp16 על GPU
import torch, time
from diffusers import WanPipeline, AutoencoderKLWan, WanTransformer3DModel
from transformers import AutoTokenizer, UMT5EncoderModel
t0 = time.time()
MID = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
text_encoder = UMT5EncoderModel.from_pretrained(MID, subfolder='text_encoder', torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MID, subfolder='tokenizer')
vae = AutoencoderKLWan.from_pretrained(MID, subfolder='vae', torch_dtype=torch.float32)
transformer = WanTransformer3DModel.from_pretrained(MID, subfolder='transformer', torch_dtype=torch.float16)
pipe = WanPipeline.from_pretrained(MID, text_encoder=text_encoder, tokenizer=tokenizer, vae=vae, transformer=transformer)
pipe.enable_model_cpu_offload()
if hasattr(pipe, 'enable_vae_tiling'): pipe.enable_vae_tiling()
print('נטען ב', round(time.time()-t0,1), 'שניות')

In [ ]:
PROMPT = "A pale goth woman, pale porcelain skin with freckles, sharp black winged eyeliner, dark plum lipstick, long straight jet-black hair with straight blunt bangs and one deep-purple streak, silver eyebrow piercing. LOCKED BODY: pronounced hourglass figure, full natural bust, narrow small waist, wide full hips, soft rounded thighs, average height, same face and same proportions in every frame. Underground poker room, green felt table, warm brass lamp light overhead, chips and cards on the table. She plays poker against her best friend, a laughing dark-haired young woman in a red flannel shirt with rolled-up sleeves. She loses the hand, reacts in amused disbelief, then she grabs the hem of her short fitted black knit dress with both hands, slowly pulls it up over her hips and waist, up past her bare midriff, then lifts it over her head with both arms raised high, revealing her black lace bra and matching black lace panties, and tosses the dress on the pile, hands on her hips, smirking. Her friend laughs and pulls the chips toward herself. Beneath the dress she wears a black lace bra and matching black lace panties. Continuous full-body motion throughout, smooth natural movement, fabric and hair moving realistically. Handheld cinematic camera, film grain, photorealistic, realistic skin texture. "
NEG = "bright tones, overexposed, static, frozen, motionless, still picture, blurred details, subtitles, style of painting, overall gray, worst quality, low quality, ugly, incomplete, extra fingers, poorly drawn hands, deformed, disfigured, misshapen limbs, fused fingers, messy background, cartoon, anime, illustration, cgi, 3d render, nudity, topless, bare breasts, male figures, brother, man at the table"
ANCHOR = "https://media.base44.com/images/public/6a901f55a4d9a9b76a095a77/e8383ce46_generated_image.png"
FRAMES = 121  # 7.5 שניות @16fps
import urllib.request
urllib.request.urlretrieve(ANCHOR, 'init.png')
from PIL import Image
Image.open('init.png')

In [ ]:
# 4) רנדר סצינה 4 + וידוא שאין שחור + שמירה
import torch, numpy as np, imageio, time, gc, io
from PIL import Image
t0 = time.time()
out = pipe(
    image=Image.open('init.png').convert('RGB'),
    prompt=PROMPT, negative_prompt=NEG,
    height=480, width=832, num_frames=FRAMES,
    num_inference_steps=20, guidance_scale=5.0, max_sequence_length=512,
    generator=torch.Generator('cpu').manual_seed(20261012),
).frames[0]
arr = np.asarray(out[0]) if isinstance(out[0], Image.Image) else np.stack([np.asarray(f) for f in out[:1]])[0]
a = np.asarray(Image.fromarray(arr).convert('RGB')).astype('float32')
print('ממוצע בהירות פריים ראשון:', round(float(a.mean()),1))
assert a.mean() > 5, 'פלט שחור — בדוק dtype'
imageio.mimwrite('s4.mp4', [np.asarray(f.convert('RGB') if isinstance(f, Image.Image) else Image.fromarray(f).convert('RGB')) for f in out], fps=16, quality=9)
Image.fromarray(np.asarray(out[0] if isinstance(out[0], Image.Image) else out[0])).save('s4_first.png')
last = out[-1] if isinstance(out[-1], Image.Image) else Image.fromarray(out[-1])
last.save('s4_last.png')
import os
print('סיום ב', round((time.time()-t0)/60,1), 'דקות | גודל:', round(os.path.getsize('s4.mp4')/1e6,1), 'MB')

In [ ]:
# 5) העלאה ל-catbox + דיווח לגשר שימי פיתוח
import requests, json
def catbox(path):
    r = requests.post('https://catbox.moe/user/api.php',
        data={'reqtype':'fileupload'}, files={'fileToUpload': open(path,'rb')}, timeout=600)
    r.raise_for_status(); return r.text
url_mp4 = catbox('s4.mp4')
url_first = catbox('s4_first.png')
url_last = catbox('s4_last.png')
print('MP4:', url_mp4)
print('FIRST:', url_first)
print('LAST:', url_last)
try:
    requests.post('https://app.base44.com/api/apps/6a901f55a4d9a9b76a095a77/backend_functions/pokerColabRelay',
        json={'action':'report','scene':'s4','url':url_mp4,'first':url_first,'last':url_last,
              'source':'colab','frames':121}, timeout=30)
    print('דווח לגשר')
except Exception as e:
    print('דיווח לגשר נכשל (הקישורים למעלה תקפים):', e)